Loading initial data

In [19]:
import pandas as pd

In [20]:
data_path = "data/cleaned_gas_monitoring.csv"
df = pd.read_csv(data_path)

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                10000 non-null  str    
 1   Temperature                10000 non-null  float64
 2   Humidity                   8072 non-null   float64
 3   CO2_InfraredSensor         10000 non-null  float64
 4   CO2_ElectroChemicalSensor  10000 non-null  float64
 5   MetalOxideSensor_Unit1     10000 non-null  float64
 6   MetalOxideSensor_Unit2     8590 non-null   float64
 7   MetalOxideSensor_Unit3     10000 non-null  float64
 8   MetalOxideSensor_Unit4     10000 non-null  float64
 9   CO_GasSensor               9166 non-null   float64
 10  Session ID                 10000 non-null  int64  
 11  HVAC Operation Mode        10000 non-null  str    
 12  Ambient Light Level        8946 non-null   str    
 13  Activity Level             10000 non-null  str    
dtypes:

In [22]:
df.isnull().sum()

Time of Day                     0
Temperature                     0
Humidity                     1928
CO2_InfraredSensor              0
CO2_ElectroChemicalSensor       0
MetalOxideSensor_Unit1          0
MetalOxideSensor_Unit2       1410
MetalOxideSensor_Unit3          0
MetalOxideSensor_Unit4          0
CO_GasSensor                  834
Session ID                      0
HVAC Operation Mode             0
Ambient Light Level          1054
Activity Level                  0
dtype: int64

Missing Values, etc still remain so I'll be cleaning these to ensure its okay

In [23]:
df.duplicated().sum()

np.int64(265)

## Removing Duplicates first;

In [24]:
df_dropped = df.drop_duplicates()
df_dropped.info()
df_dropped.shape

<class 'pandas.DataFrame'>
Index: 9735 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                9735 non-null   str    
 1   Temperature                9735 non-null   float64
 2   Humidity                   7824 non-null   float64
 3   CO2_InfraredSensor         9735 non-null   float64
 4   CO2_ElectroChemicalSensor  9735 non-null   float64
 5   MetalOxideSensor_Unit1     9735 non-null   float64
 6   MetalOxideSensor_Unit2     8333 non-null   float64
 7   MetalOxideSensor_Unit3     9735 non-null   float64
 8   MetalOxideSensor_Unit4     9735 non-null   float64
 9   CO_GasSensor               8906 non-null   float64
 10  Session ID                 9735 non-null   int64  
 11  HVAC Operation Mode        9735 non-null   str    
 12  Ambient Light Level        8685 non-null   str    
 13  Activity Level             9735 non-null   str    
dtypes: float

(9735, 14)

## Identifying numerical and categorical cols

In [32]:
numerical_features = ['Temperature', 'Humidity', 'CO2_InfraredSensor', 'CO2_ElectroChemicalSensor', 'MetalOxideSensor_Unit1', 'MetalOxideSensor_Unit2', 'MetalOxideSensor_Unit3', 'MetalOxideSensor_Unit4', 'CO_GasSensor'] # session id not included as it is not a feature for modeling
categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level', 'Activity Level'] # activity level to remove later because its our target variable

## Handling null values
1. Numerical columns, we using Median 
2. Categorical column (ambient light level) by choosing the light level referencing the time of day for missing data. More realistic

In [27]:
for col in ['Humidity', 'MetalOxideSensor_Unit2', 'CO_GasSensor']:
    if col in numerical_features:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

if 'Ambient Light Level' in categorical_features:
    # Impute missing light levels using the mode of their specific Time of Day
    df['Ambient Light Level'] = df.groupby('Time of Day')['Ambient Light Level'].transform(
        lambda x: x.fillna(x.mode()[0])
    )

print("\nMissing values after imputation:")
print(df[numerical_features + categorical_features].isnull().sum())


Missing values after imputation:
Temperature                  0
Humidity                     0
CO2_InfraredSensor           0
CO2_ElectroChemicalSensor    0
MetalOxideSensor_Unit1       0
MetalOxideSensor_Unit2       0
MetalOxideSensor_Unit3       0
MetalOxideSensor_Unit4       0
CO_GasSensor                 0
Time of Day                  0
HVAC Operation Mode          0
Ambient Light Level          0
Activity Level               0
dtype: int64


## Train Test Split (before fit scaling/encoding)


In [30]:
from sklearn.model_selection import train_test_split

In [28]:
activity_mapping = {
    'Low Activity': 0,
    'Moderate Activity': 1,
    'High Activity': 2
}

y = df['Activity Level'].map(activity_mapping).to_numpy()

In [29]:
X_raw = df.drop(columns=['Activity Level', 'Session ID'])

In [31]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Encoding

ASSUMPTION: Our problem statement ask us to find relationships. So I will encode the stuff with nominal encoding (no relationships/hierarchy), and assume nothing about them.

I will use One Hot Encoding on everything first, since it is a form of encoding that does not explictly say something has a relationship/hiererachy, etc)

But for activity level, i will add a hierarchy (because it is important)

In [33]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
X_train_cat_encoded = encoder.fit_transform(X_train_raw[categorical_features])
X_test_cat_encoded = encoder.transform(X_test_raw[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_feature_names, index=X_train_raw.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_feature_names, index=X_test_raw.index)


In [34]:
X_train_cat_df.head()

,Time of Day_evening,Time of Day_morning,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
9490,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8825,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8145,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
7354,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7723,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [35]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_raw[numerical_features])

X_test_num_scaled = scaler.transform(X_test_raw[numerical_features])

X_train_num_df = pd.DataFrame(X_train_num_scaled, columns=numerical_features, index=X_train_raw.index)
X_test_num_df = pd.DataFrame(X_test_num_scaled, columns=numerical_features, index=X_test_raw.index)

In [37]:
X_train_num_df.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor
9490,0.334890,0.040282,-0.103489,-1.366623,0.155770,0.184965,-0.599483,-0.249664,-0.349569
8825,0.751603,-0.046740,1.894198,-1.248529,0.025715,0.354781,-0.003973,0.257759,-0.349569
8145,1.254804,0.006322,-0.082515,-1.145318,-0.779956,-1.096242,-0.346177,-0.288881,0.980855
7354,0.739810,0.040282,0.374523,-1.245661,-1.076695,-2.007138,-0.778547,-0.993211,2.311280
7723,1.553580,-0.010127,0.033737,-1.415704,-0.090730,-0.854412,-0.231061,-0.422101,0.980855


In [38]:
X_train_final = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

In [42]:
X_train_final.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Time of Day_evening,...,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
9490,0.334890,0.040282,-0.103489,-1.366623,0.155770,0.184965,-0.599483,-0.249664,-0.349569,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8825,0.751603,-0.046740,1.894198,-1.248529,0.025715,0.354781,-0.003973,0.257759,-0.349569,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8145,1.254804,0.006322,-0.082515,-1.145318,-0.779956,-1.096242,-0.346177,-0.288881,0.980855,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
7354,0.739810,0.040282,0.374523,-1.245661,-1.076695,-2.007138,-0.778547,-0.993211,2.311280,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7723,1.553580,-0.010127,0.033737,-1.415704,-0.090730,-0.854412,-0.231061,-0.422101,0.980855,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [41]:
X_test_final.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Time of Day_evening,...,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
3212,0.433172,-0.273317,0.941336,-0.081330,-2.248504,-0.418269,-0.115987,0.029340,-0.349569,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7687,-0.243005,0.043996,-0.158922,-1.030434,-0.493288,-0.052810,-0.136554,-0.267895,-0.349569,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5573,1.396330,-0.166661,0.199879,-0.406847,-0.413117,-0.710728,-0.179796,-0.170870,0.980855,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
6857,-1.713295,-0.456382,-1.072688,1.368736,0.632359,2.230572,1.387833,1.901855,-0.349569,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8842,0.099015,0.258369,0.630926,-1.263375,-0.476439,-0.052810,-0.281867,-0.617724,-0.349569,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [45]:
print(y_train[:5])
print(y_test[:5])

[0 2 0 0 0]
[0 1 0 1 0]


# Cleaning Finished
# Moving to ML

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
